#### **`bedtools Intersect intervals`**
To find overlapping intervals in various ways

| Parameter | Description |
| --- | --- |
| Combined or separate output files | To save intersect data in a single file or one for each intersection |
| Calculation based on strandedness? | Can choose to restrict to overlap occuring on the same or opposite or either strand |
| What should be written to the output file? | Whether columns in file a or file b or both is prioritize and whether to set non-intersect row with null |


In [31]:
from pathlib import Path
import subprocess

exons = Path("/Users/nguyuling/Galaxy-Sequence-Analysis/datasets/4104428/UCSC-hg38-chr22-Coding-Exons.bed")
snps = Path("/Users/nguyuling/Galaxy-Sequence-Analysis/datasets/4104428/UCSC-hg38-chr22-dbSNP153-Whole-Gene-SNPs.bed")
intersect = Path("/Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/intersect.bed")

result = subprocess.run(
    [
        "bedtools",
        "intersect",
        "-a",
        str(exons),
        "-b",
        str(snps),
        "-wa",
        "-wb",
    ],
    check=True,
    capture_output=True,
    text=True,
)

intersect.write_text(result.stdout)
intersection_lines = result.stdout.splitlines()

print(f"Found {len(intersection_lines)} intersecting interval pairs.")
print(f"Results written to: {intersect}")


Found 6719 intersecting interval pairs.
Results written to: /Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/intersect.bed


#### **`Datamash`**

In [32]:
import subprocess

datamash = intersect.with_name("datamash.bed")
sorted_intersect = "\n".join(
    sorted(
        intersection_lines,
        key=lambda line: line.split("\t")[3],
    )
)

counts = subprocess.run(
    ["datamash", "-g", "4", "countunique", "10"],
    input=sorted_intersect,
    check=True,
    capture_output=True,
    text=True,
)

datamash.write_text("exon\tunique_snp_count\n" + counts.stdout)
count_lines = counts.stdout.splitlines()

print(f"Counted unique SNPs for {len(count_lines)} exons.")
print(f"Results written to: {datamash}")


Counted unique SNPs for 4242 exons.
Results written to: /Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/datamash.bed


#### **`Sort`**

In [33]:
sort = datamash.with_name("sort.bed")

sorted_counts = subprocess.run(
    ["sort", "-t", "\t", "-k2,2nr"],
    input="\n".join(count_lines) + "\n",
    check=True,
    capture_output=True,
    text=True,
)

sort.write_text(
    "exon\tunique_snp_count\n" + sorted_counts.stdout
)
sorted_count_lines = sorted_counts.stdout.splitlines()

print(f"Sorted {len(sorted_count_lines)} exon counts in descending order.")
print(f"Results written to: {sort}")
print("exon\tunique_snp_count")
print("\n".join(sorted_count_lines[:5]))


Sorted 4242 exon counts in descending order.
Results written to: /Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/sort.bed
exon	unique_snp_count
ENST00000253255.7_cds_0_0_chr22_46256561_r	27
ENST00000648057.3_cds_0_0_chr22_50546244_f	26
ENST00000327423.11_cds_5_0_chr22_31712083_r	20
ENST00000302097.3_cds_0_0_chr22_22514002_r	14
ENST00000216268.6_cds_1_0_chr22_49883663_f	13


#### **`Select first`**

In [35]:
select_first = sort.with_name("select-first.bed")
sort_lines = sort.read_text().splitlines()
header, sorted_exon_lines = sort_lines[0], sort_lines[1:]

selected = subprocess.run(
    ["head", "-n", "5"],
    input="\n".join(sorted_exon_lines) + "\n",
    check=True,
    capture_output=True,
    text=True,
)

selected_lines = selected.stdout.splitlines()
select_first.write_text(header + "\n" + selected.stdout)

print(f"Selected {len(selected_lines)} exons with the most SNPs.")
print(f"Results written to: {select_first}")

Selected 5 exons with the most SNPs.
Results written to: /Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/select-first.bed


#### **`Compare two Datasets`**

In [36]:
compare_two_datasets = select_first.with_name("compare-two-datasets.bed")
selected_exons = {
    line.split("\t", 1)[0]
    for line in select_first.read_text().splitlines()[1:]
}

matching_rows = [
    line
    for line in exons.read_text().splitlines()
    if len(line.split("\t")) >= 4 and line.split("\t")[3] in selected_exons
]

compare_two_datasets.write_text("\n".join(matching_rows) + "\n")

print(f"Found {len(matching_rows)} matching rows in the first dataset.")
print(f"Results written to: {compare_two_datasets}")
print("\n".join(matching_rows))


Found 5 matching rows in the first dataset.
Results written to: /Users/nguyuling/Galaxy-Sequence-Analysis/basics-genomics/compare-two-datasets.bed
chr22	22514001	22515630	ENST00000302097.3_cds_0_0_chr22_22514002_r	0	-
chr22	31712082	31717291	ENST00000327423.11_cds_5_0_chr22_31712083_r	0	-
chr22	46256560	46263322	ENST00000253255.7_cds_0_0_chr22_46256561_r	0	-
chr22	49883662	49887178	ENST00000216268.6_cds_1_0_chr22_49883663_f	0	+
chr22	50546243	50549951	ENST00000648057.3_cds_0_0_chr22_50546244_f	0	+
